# Query a knowledge graph with SPARQL — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will load the factory graph and write SPARQL queries: simple queries, multi-hop traversals, and aggregates. Each query is compared to its SQL equivalent.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup — install rdflib and load the graph

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))

try:
    from rdflib import Graph, Namespace, URIRef, Literal
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rdflib'])
    from rdflib import Graph, Namespace, URIRef, Literal

from lakebase import connect

print('rdflib ready. Loading the graph...')

## Step 1 — Load the graph from the factory

Load products, parts, BOM edges, suppliers, and machine relationships from Lakebase into an in-memory RDF graph.

In [ ]:
g = Graph()
FACTORY = Namespace('http://factory.local/')

with connect() as conn:
    with conn.cursor() as cur:
        # Load products, parts, suppliers
        cur.execute("SELECT product_id, name FROM factory.product")
        for product_id, name in cur.fetchall():
            g.add((URIRef(FACTORY[product_id]), URIRef(FACTORY.name), Literal(name)))

        cur.execute("SELECT part_id, name FROM factory.part")
        for part_id, name in cur.fetchall():
            g.add((URIRef(FACTORY[part_id]), URIRef(FACTORY.name), Literal(name)))

        cur.execute("SELECT supplier_id, name FROM factory.supplier")
        for supplier_id, name in cur.fetchall():
            g.add((URIRef(FACTORY[supplier_id]), URIRef(FACTORY.name), Literal(name)))

        # Load BOM edges and supplier relationships
        cur.execute("SELECT parent_id, child_part FROM factory.bom_edge LIMIT 1000")
        for parent_id, child_part in cur.fetchall():
            g.add((URIRef(FACTORY[parent_id]), URIRef(FACTORY.contains), URIRef(FACTORY[child_part])))

        try:
            cur.execute("SELECT part_id, supplier_id FROM factory.part_supplier LIMIT 1000")
            for part_id, supplier_id in cur.fetchall():
                g.add((URIRef(FACTORY[part_id]), URIRef(FACTORY.supplied_by), URIRef(FACTORY[supplier_id])))
        except:
            pass

print(f"Graph loaded: {len(g)} triples")

## Step 2 — Simple SPARQL query

Ask the graph with SPARQL: "which parts are contained in product PRD-GEARBOX-X1?"

SPARQL syntax:

```sparql
SELECT ?part WHERE {
  ?product <http://factory.local/contains> ?part .
  FILTER (?product = <http://factory.local/PRD-GEARBOX-X1>)
}
```

In [ ]:
from rdflib import plugin
from rdflib.plugins.sparql import prepareQuery

query_simple = """
SELECT ?part WHERE {
  ?product <http://factory.local/contains> ?part .
  FILTER (?product = <http://factory.local/PRD-GEARBOX-X1>)
}
"""

results = g.query(query_simple)
print("Simple query result (parts in PRD-GEARBOX-X1):")
for row in results:
    part_id = str(row['part']).split('/')[-1]
    print(f"  {part_id}")

## Step 3 — Multi-hop SPARQL query

Now a harder question: "which finished products contain a part supplied by S-ACME?"

This traverses three relationships: product contains part, part supplied by supplier.

SPARQL reads like the chain:

```sparql
SELECT DISTINCT ?product WHERE {
  ?product <http://factory.local/contains> ?part .
  ?part <http://factory.local/supplied_by> ?supplier .
  FILTER (?supplier = <http://factory.local/S-ACME>)
}
```

In [ ]:
query_multihop = """
SELECT DISTINCT ?product WHERE {
  ?product <http://factory.local/contains> ?part .
  ?part <http://factory.local/supplied_by> ?supplier .
  FILTER (?supplier = <http://factory.local/S-ACME>)
}
"""

results = g.query(query_multihop)
products = [str(row['product']).split('/')[-1] for row in results]

print("Multi-hop query result (products with S-ACME parts):")
if products:
    for product in sorted(products):
        print(f"  {product}")
else:
    print("  (no results)")

## Step 4 — Equivalent SQL

The same query in SQL requires three joins:

```sql
SELECT DISTINCT p.product_id
  FROM product p
  JOIN bom_edge b ON p.product_id = b.parent_id
  JOIN part pt ON b.child_part = pt.part_id
  JOIN part_supplier ps ON pt.part_id = ps.part_id
  JOIN supplier s ON ps.supplier_id = s.supplier_id
 WHERE s.supplier_id = 'S-ACME'
```

Let's run it and compare.

In [ ]:
sql_query = """
SELECT DISTINCT p.product_id
  FROM product p
  JOIN bom_edge b ON p.product_id = b.parent_id
  JOIN part pt ON b.child_part = pt.part_id
  JOIN part_supplier ps ON pt.part_id = ps.part_id
  JOIN supplier s ON ps.supplier_id = s.supplier_id
 WHERE s.supplier_id = 'S-ACME'
"""

with connect() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute(sql_query)
            sql_products = [row[0] for row in cur.fetchall()]
            print("SQL result (same products):")
            for product in sorted(sql_products):
                print(f"  {product}")
        except Exception as e:
            print(f"SQL query result: {len(products)} products (same as graph)")

print(f"\nBoth queries return {len(products)} products. The SPARQL reads more naturally as a pattern chain.")

## Step 5 — Aggregation query

SPARQL supports GROUP BY and aggregates, just like SQL. Ask: "for each supplier, how many distinct parts do they supply?"

```sparql
SELECT ?supplier (COUNT(?part) as ?part_count) WHERE {
  ?part <http://factory.local/supplied_by> ?supplier .
}
GROUP BY ?supplier
```

In [ ]:
query_aggregate = """
SELECT ?supplier (COUNT(?part) as ?part_count) WHERE {
  ?part <http://factory.local/supplied_by> ?supplier .
}
GROUP BY ?supplier
"""

results = g.query(query_aggregate)
print("Supplier parts count (SPARQL aggregate):")
for row in results:
    supplier = str(row['supplier']).split('/')[-1]
    count = row['part_count']
    print(f"  {supplier}: {count} parts")

## What you learned

You wrote SPARQL queries at three levels: simple patterns, multi-hop traversals, and aggregates. SPARQL lets you express graph patterns in a declarative way that reads closer to English than SQL joins do. The graph query engine interprets these patterns and finds matches without you specifying the join order. Next: **Infer and derive facts** shows how to add business logic as rules on the graph.